# PTM-LLaMA Model Training

Fine-tune `GreatCaptainNemo/ProLLaMA_Stage_1` with a LoRA adapter to predict post-translational modification (PTM) sites from short peptide sequences. The model is instruction-tuned to handle three PTM types in a single adapter: methylation, phosphorylation, and ubiquitination. The PTM type to predict is selected at inference time by the instruction prompt; the output format is `Sites=<R5,D12,...>` regardless of PTM type.

Implemented with `transformers`, `peft`, and `trl.SFTTrainer`. The notebook is intended to run in Google Colab on a GPU runtime (A100 preferred; L4 and T4 also work). The merged model is pushed to the Hugging Face Hub for the evaluation notebook to consume.

## 1. Environment Setup

Install the required libraries. Major versions are pinned for reproducibility. If Colab prompts for a runtime restart (`Runtime > Restart runtime`), do so and resume from the next cell.

In [ ]:
!pip install -q -U \
    "transformers>=4.44,<4.50" \
    "peft>=0.11" \
    "trl>=0.9,<0.12" \
    "accelerate>=0.30" \
    "bitsandbytes>=0.43" \
    "datasets>=2.20" \
    "huggingface_hub>=0.24" \
    scikit-learn matplotlib "pandas<3" sentencepiece

# peft >=0.11 errors out if torchao is installed at an incompatible version.
# Colab ships torchao==0.10.0 pre-installed; we don't use it, so remove it.
!pip uninstall -y -q torchao

In [ ]:
import os, json, math, gc, hashlib

# Reduce CUDA allocator fragmentation. Variable-length batches during SFT can
# leave the reserved-but-unallocated pool fragmented enough to fail a large
# contiguous allocation mid-training even with substantial free VRAM.
# Must be set BEFORE `import torch` so torch reads it at CUDA init.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    EarlyStoppingCallback,
)
from peft import LoraConfig, AutoPeftModelForCausalLM
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from huggingface_hub import HfApi, login

print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Configuration

All hyperparameters are surfaced in this cell. The PTM-type instruction templates and the protein-level split parameters are also defined here so the evaluation notebook can re-derive identical partitions from the same source CSV and seed.

The training data is drawn from `datasets/all_ptm_sites_site_level.csv`, a long-format file with one row per annotated PTM site across three PTM types (Methylation, Phosphorylation, Ubiquitination). The split into train, calibration, and test sets is performed in this notebook at the protein level — every annotation of a given UniProt ID is assigned to a single split, eliminating sequence-level leakage. The evaluation notebook re-derives the same calibration and test partitions deterministically from the same source CSV and random seed.

In [ ]:
BASE_MODEL = 'GreatCaptainNemo/ProLLaMA_Stage_1'

PTM_DATA_CSV = 'datasets/all_ptm_sites_site_level.csv'
SPLIT_SEED = 42
TRAIN_RATIO = 0.80
CAL_RATIO = 0.10
TEST_RATIO = 0.10
VAL_SIZE = 0.10  # fraction of the train partition reserved for early-stopping validation

WINDOW_SIZE = 21
STRIDE = 5

# Per-PTM-type cap on the negative:positive window ratio in the training fold.
# A window is "positive" if its target contains at least one in-window site, and
# "negative" if its target is `Sites=<>`. The cap is applied independently per
# PTM type; PTM types whose natural ratio is already <= NEG_TO_POS_RATIO are
# left untouched. Set to None to disable subsampling entirely.
NEG_TO_POS_RATIO = 3

PTM_INSTRUCTIONS = {
    'Methylation':     '[Predict the methylation sites given the peptide sequence]',
    'Phosphorylation': '[Predict the phosphorylation sites given the peptide sequence]',
    'Ubiquitination':  '[Predict the ubiquitination sites given the peptide sequence]',
}

OUTPUT_DIR = '/content/drive/MyDrive/ptm-llama/saves/ptm-llama'
MERGED_DIR = OUTPUT_DIR + '-merged'

HF_REPO_ID = 'jbenbudd/ptm-llama'

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'down_proj', 'up_proj',
]

LEARNING_RATE = 1e-4
NUM_TRAIN_EPOCHS = 8
# Hard cap on optimizer steps, sized well below Colab's ~24h session ceiling
# and just past the v3.1 plateau (eval loss ~0.140 by step 8000, marginal drift
# thereafter). `max_steps` overrides `num_train_epochs` for the LR schedule so
# the cosine decays to zero by MAX_STEPS - i.e. late steps become a proper
# cooldown phase rather than staying at near-peak LR.
MAX_STEPS = 10000
PER_DEVICE_TRAIN_BATCH_SIZE = 32
PER_DEVICE_EVAL_BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = 4
LR_SCHEDULER = 'cosine'
WARMUP_STEPS = 200
MAX_SEQ_LENGTH = 2048
EVAL_STEPS = 500
SAVE_STEPS = 500
LOGGING_STEPS = 20
SAVE_TOTAL_LIMIT = 3
EARLY_STOPPING_PATIENCE = 5
MAX_GRAD_NORM = 0.5

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Authenticate to Hugging Face

In Colab, store your token in `Secrets > HF_API_TOKEN` (the lock icon in the left sidebar). Outside Colab, export `HF_API_TOKEN` in your shell.

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_API_TOKEN')
except:
    HF_TOKEN = os.environ.get('HF_API_TOKEN')

assert HF_TOKEN, 'No HF token found. Set HF_API_TOKEN in Colab Secrets or env vars.'
login(token=HF_TOKEN)
print('Logged in to Hugging Face.')

## 4. Data Loading and Protein-Level Split

The source CSV is in long format, one row per annotated PTM site:

| uniprot_id | PTM_site | PTM_Type | protein_sequence | binary_mask | literature_reference |
|---|---|---|---|---|---|

The pipeline performs three transformations:

1. **Group by `(uniprot_id, PTM_Type)`** — within each group, the protein sequence and binary mask are constant (the mask encodes all annotated sites of that PTM type on that protein). One record per unique combination is retained.
2. **Protein-level split** — unique `uniprot_id`s are shuffled with `SPLIT_SEED` and partitioned 80 / 10 / 10 into train, calibration, and test. All annotations of a given protein are assigned to a single split, preventing sequence-level leakage across splits.
3. **Window expansion** — for each `(protein, PTM_type)` record in the train partition, a 21-residue window is slid across the protein sequence with stride 5 (and a tail window so the final residues are covered). Each window is converted into one instruction-tuned training example whose instruction is selected by PTM type and whose target is the comma-separated list of in-window sites of that PTM type. Windows containing no site of that PTM type are retained as in-context negatives (target `Sites=<>`).

Only the train partition is expanded in this notebook; the calibration and test partitions are consumed by the evaluation notebook.

In [ ]:
df = pd.read_csv(PTM_DATA_CSV)
print(f'Loaded {len(df):,} annotated sites from {PTM_DATA_CSV}')
print('Columns:', list(df.columns))
print()
print('Sites per PTM type:')
print(df['PTM_Type'].value_counts().to_string())
print()
print(f'Unique proteins: {df["uniprot_id"].nunique():,}')
df.head(3)

In [ ]:
# Group by (uniprot_id, PTM_Type): retain the protein sequence and binary mask.
# The mask within a group encodes all sites of that PTM type on that protein,
# so a single representative row per group is sufficient for window expansion.
groups = (
    df.drop_duplicates(subset=['uniprot_id', 'PTM_Type'])
      [['uniprot_id', 'PTM_Type', 'protein_sequence', 'binary_mask']]
      .reset_index(drop=True)
)

# Validate sequence consistency within each uniprot_id (sequence must not vary
# across PTM-type rows for the same protein).
seq_per_id = df.groupby('uniprot_id')['protein_sequence'].nunique()
inconsistent = seq_per_id[seq_per_id > 1]
if len(inconsistent) > 0:
    print(f'WARNING: {len(inconsistent)} uniprot_ids have multiple distinct sequences. First few:')
    print(inconsistent.head().to_string())
    groups = groups.drop_duplicates(subset=['uniprot_id', 'PTM_Type'], keep='first')

print(f'Unique (uniprot_id, PTM_Type) combinations: {len(groups):,}')
print()
print('Combinations per PTM type:')
print(groups['PTM_Type'].value_counts().to_string())

In [ ]:
# Protein-level split with a fixed seed.
unique_proteins = np.array(sorted(df['uniprot_id'].unique()))
rng = np.random.default_rng(SPLIT_SEED)
shuffled = rng.permutation(unique_proteins)

n_total = len(shuffled)
n_train = int(n_total * TRAIN_RATIO)
n_cal = int(n_total * CAL_RATIO)
# Test gets the remainder so the splits sum exactly to n_total.

train_proteins = set(shuffled[:n_train])
cal_proteins   = set(shuffled[n_train:n_train + n_cal])
test_proteins  = set(shuffled[n_train + n_cal:])

assert not (train_proteins & cal_proteins)
assert not (train_proteins & test_proteins)
assert not (cal_proteins & test_proteins)

print(f'Protein-level split (seed = {SPLIT_SEED}, ratios = {TRAIN_RATIO}/{CAL_RATIO}/{TEST_RATIO}):')
print(f'  train:       {len(train_proteins):,} proteins')
print(f'  calibration: {len(cal_proteins):,} proteins')
print(f'  test:        {len(test_proteins):,} proteins')
print(f'  total:       {n_total:,} proteins')

# Persist the split assignment for documentation and reproducibility.
splits_path = os.path.join(OUTPUT_DIR, 'protein_splits.json')
with open(splits_path, 'w') as f:
    json.dump({
        'split_seed': SPLIT_SEED,
        'ratios': {'train': TRAIN_RATIO, 'calibration': CAL_RATIO, 'test': TEST_RATIO},
        'train_proteins': sorted(train_proteins),
        'calibration_proteins': sorted(cal_proteins),
        'test_proteins': sorted(test_proteins),
    }, f)
print(f'\nSaved split assignment to {splits_path}')

In [ ]:
def build_windows(seq: str, w: int = WINDOW_SIZE, s: int = STRIDE):
    L = len(seq)
    if L <= w:
        return [(0, seq)]
    starts = list(range(0, L - w + 1, s))
    if starts[-1] + w < L:
        starts.append(L - w)
    return [(start, seq[start:start + w]) for start in starts]


def build_examples_for_group(uniprot_id: str, ptm_type: str, sequence: str, binary_mask: str):
    """Expand one (protein, PTM_type) record into per-window instruction examples."""
    if pd.isna(sequence) or pd.isna(binary_mask):
        return []
    sequence = str(sequence)
    binary_mask = str(binary_mask)
    if len(sequence) != len(binary_mask):
        return []
    instruction = PTM_INSTRUCTIONS.get(ptm_type)
    if instruction is None:
        return []
    examples = []
    for start, window_seq in build_windows(sequence):
        window_mask = binary_mask[start:start + len(window_seq)]
        sites = [
            f'{window_seq[j]}{j + 1}'
            for j, ch in enumerate(window_mask) if ch == '1'
        ]
        examples.append({
            'instruction': instruction,
            'input':  f'Seq=<{window_seq}>',
            'output': f'Sites=<{",".join(sites)}>',
            'uniprot_id': uniprot_id,
            'ptm_type': ptm_type,
        })
    return examples


# Expand the train partition. (Calibration and test are deferred to the eval notebook.)
train_records = []
for row in groups.itertuples(index=False):
    if row.uniprot_id not in train_proteins:
        continue
    train_records.extend(build_examples_for_group(
        row.uniprot_id, row.PTM_Type, row.protein_sequence, row.binary_mask,
    ))

print(f'Built {len(train_records):,} training windows from the train partition.')
print()
print('Windows per PTM type:')
ptm_counts = pd.Series([r['ptm_type'] for r in train_records]).value_counts()
print(ptm_counts.to_string())
print()
n_positive = sum(1 for r in train_records if r['output'] != 'Sites=<>')
print(f'Windows with at least one in-window site (positive): {n_positive:,} ({n_positive / len(train_records) * 100:.1f}%)')
print(f'Windows with no in-window site (negative):           {len(train_records) - n_positive:,} ({(1 - n_positive / len(train_records)) * 100:.1f}%)')
print()
print('Sample record:')
print(json.dumps(train_records[0], indent=2))

## 5. Train/Validation Split and Per-PTM-Type Negative Subsampling

Two steps in this section:

1. **Train/validation split.** A small fraction of the train partition is reserved as a validation set for early stopping. The split is at the protein level via `GroupShuffleSplit` keyed on `uniprot_id`, so no protein appears in both train and validation.
2. **Per-PTM-type negative subsampling.** Site prevalence varies by orders of magnitude across PTM types (Methylation ≪ Ubiquitination ≪ Phosphorylation), so the unfiltered sliding-window expansion produces a heavily negative-skewed dataset for the rarer PTM types — enough to make the model collapse to `Sites=<>` for those instructions. To address this, after the train/validation split we cap the negative:positive window ratio to `NEG_TO_POS_RATIO` **per PTM type** by randomly subsampling the negatives. The cap is only applied where it bites; PTM types whose natural ratio is already at or below the cap are left untouched.

   The subsampling is applied to **both the training fold and the validation fold**, with independent seeds. The training fold is balanced so the model sees enough positive signal at SGD time. The validation fold is balanced because the natural distribution is dominated by `Sites=<>` targets, which means `eval_loss` on a natural-distribution val fold mostly measures the model's ability to memorize the empty prior rather than its ability to discriminate sites — and that is exactly the wrong signal for early stopping in this task. A class-balanced val fold makes `eval_loss` a faithful proxy for site-discrimination quality, which is what we want `EarlyStoppingCallback` to track.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SPLIT_SEED)
groups_arr = [r['uniprot_id'] for r in train_records]
train_idx, val_idx = next(gss.split(train_records, groups=groups_arr))
train_records_actual = [train_records[i] for i in train_idx]
val_records          = [train_records[i] for i in val_idx]

actual_train_proteins = {r['uniprot_id'] for r in train_records_actual}
val_proteins          = {r['uniprot_id'] for r in val_records}
leak = actual_train_proteins & val_proteins
print(f'Train (pre-balance):      {len(train_records_actual):,} windows | {len(actual_train_proteins):,} proteins')
print(f'Validation (pre-balance): {len(val_records):,} windows | {len(val_proteins):,} proteins')
print(f'Train/val protein overlap: {len(leak)} (must be 0)')
assert not leak


def balance_negatives_by_ptm(records, ratio, rng):
    """Cap negatives:positives per PTM type by random subsampling without replacement."""
    by_ptm: dict = {}
    for r in records:
        by_ptm.setdefault(r['ptm_type'], []).append(r)
    balanced = []
    rows = []
    for ptm in sorted(by_ptm.keys()):
        recs = by_ptm[ptm]
        pos = [r for r in recs if r['output'] != 'Sites=<>']
        neg = [r for r in recs if r['output'] == 'Sites=<>']
        n_pos = len(pos)
        n_neg_in = len(neg)
        cap = ratio * n_pos
        if n_pos > 0 and n_neg_in > cap:
            keep_idx = rng.choice(n_neg_in, size=cap, replace=False)
            neg = [neg[i] for i in keep_idx]
        rows.append((ptm, n_pos, n_neg_in, len(neg)))
        balanced.extend(pos)
        balanced.extend(neg)
    shuffle_idx = rng.permutation(len(balanced))
    return [balanced[i] for i in shuffle_idx], rows


def print_balance_table(label, rows):
    print(f'\n{label}:')
    print(f'  {"PTM type":<18} {"pos":>10} {"neg in":>12} {"neg out":>12} {"ratio":>8}')
    for ptm, n_pos, n_neg_in, n_neg_out in rows:
        ratio_str = f'{(n_neg_out / n_pos):.2f}' if n_pos > 0 else 'inf'
        print(f'  {ptm:<18} {n_pos:>10,} {n_neg_in:>12,} {n_neg_out:>12,} {ratio_str:>8}')


# Per-PTM-type negative subsampling on both the training fold and the validation
# fold. The validation fold is balanced because the natural distribution is
# dominated by `Sites=<>` targets, which makes `eval_loss` mostly measure the
# empty-prior memorization rather than site discrimination - the wrong signal
# for early stopping. Independent seeds are used so the two subsamples are not
# coupled.
if NEG_TO_POS_RATIO is not None:
    print(f'\nNegative subsampling (cap neg:pos = {NEG_TO_POS_RATIO}:1 per PTM type):')
    train_rng = np.random.default_rng(SPLIT_SEED)
    val_rng   = np.random.default_rng(SPLIT_SEED + 1)
    train_records_actual, train_rows = balance_negatives_by_ptm(
        train_records_actual, NEG_TO_POS_RATIO, train_rng,
    )
    val_records, val_rows = balance_negatives_by_ptm(
        val_records, NEG_TO_POS_RATIO, val_rng,
    )
    print_balance_table(f'Train fold (seed = {SPLIT_SEED})', train_rows)
    print_balance_table(f'Validation fold (seed = {SPLIT_SEED + 1})', val_rows)
    print(f'\nTrain (post-balance):      {len(train_records_actual):,} windows')
    print(f'Validation (post-balance): {len(val_records):,} windows')
else:
    print('\nNegative subsampling disabled (NEG_TO_POS_RATIO is None).')

## 6. Tokenizer + Dataset Construction

Each record is rendered into an Alpaca-style training string. The `### Response:` marker is used by `DataCollatorForCompletionOnlyLM` to mask everything before the response with `-100`, so loss is computed only on the answer tokens. Because SentencePiece tokenization is context-dependent, the marker is passed to the collator as a sequence of token IDs derived from in-context encoding rather than as a raw string.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = 'right'

ALPACA_TEMPLATE = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{instruction}\n{input}\n\n'
    '### Response:\n{output}'
)

def to_text(rec):
    return ALPACA_TEMPLATE.format(**rec) + tokenizer.eos_token

train_ds = Dataset.from_list([{'text': to_text(r)} for r in train_records_actual])
val_ds   = Dataset.from_list([{'text': to_text(r)} for r in val_records])

print('Example training text:')
print('-' * 60)
print(train_ds[0]['text'])
print('-' * 60)

## 7. Load Model and Configure LoRA

The base model is loaded in bfloat16 with `device_map='auto'` and gradient checkpointing enabled. The LoRA configuration targets the attention projections (`q`, `k`, `v`, `o`) and the MLP projections (`gate`, `down`, `up`).

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=LORA_TARGET_MODULES,
)
print(peft_config)

In [ ]:
# GPU + model dtype sanity check.
print('=== GPU ===')
print(f'  device                : {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'  total VRAM            : {props.total_memory / 1e9:.1f} GB')
print(f'  compute capability    : {props.major}.{props.minor}')
print(f'  bf16 supported        : {torch.cuda.is_bf16_supported()}')
print()
print('=== Model ===')
sample_param = next(model.parameters())
print(f'  sample param dtype    : {sample_param.dtype}')
print(f'  sample param device   : {sample_param.device}')
print(f'  config.use_cache      : {model.config.use_cache}')
print(f'  flash attention impl  : {getattr(model.config, "_attn_implementation", "n/a")}')
print()
print('=== Allocated memory ===')
print(f'  allocated             : {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print(f'  reserved              : {torch.cuda.memory_reserved() / 1e9:.2f} GB')

## 8. Trainer Configuration

**Training loss vs. evaluation loss.** Training loss is computed on the partition the model is actively updated on. Evaluation loss is computed on held-out windows (`val_ds`) the model does not see during gradient updates and is therefore the operative measure of generalization.

**Early stopping.** `EarlyStoppingCallback(patience=EARLY_STOPPING_PATIENCE)` terminates training if `eval_loss` fails to improve for that many consecutive evaluations. Combined with `load_best_model_at_end=True`, the final saved model is the checkpoint with the lowest `eval_loss` observed during training. The patience is set higher than a typical default because LoRA + instruction tuning on this corpus shows transient `eval_loss` oscillations during the early phase that recover on their own — a stricter patience kills training before the model has done meaningful work.

**Learning rate schedule.** Linear warmup over `WARMUP_STEPS` steps ramps the learning rate from zero to its peak gradually so the optimizer does not punch through into divergence at the warmup peak; cosine decay then reduces the learning rate toward the end of training so late updates refine the loss surface rather than oscillate across it. `MAX_GRAD_NORM` clips per-step gradients as a further safety net against spikes.

**Wall-clock cap and checkpoint persistence.** `MAX_STEPS` sets a hard ceiling on the number of optimizer steps and is sized well below the runtime session limit of the training environment. It overrides `NUM_TRAIN_EPOCHS` for the LR schedule horizon, so the cosine decays to zero by `MAX_STEPS` and the tail of training becomes a proper cooldown phase rather than staying at near-peak LR. `OUTPUT_DIR` points at a persistent mount (e.g. Google Drive) so that all intermediate checkpoints, the best checkpoint, the merged model, and the training logs survive a runtime disconnect. Early stopping remains active and will still fire if `eval_loss` plateaus before `MAX_STEPS` is reached.

**Training distribution.** Each (protein, PTM_type) record in the train partition contributes all of its sliding windows, including windows with no in-window site of that PTM type (target `Sites=<>`). After the protein-level train/validation split, **both folds** are class-balanced per PTM type: negatives in excess of `NEG_TO_POS_RATIO`×positives are randomly subsampled (deterministically, with seeds derived from `SPLIT_SEED`). PTM types whose natural negative:positive ratio is already at or below the cap are left untouched. This prevents rare PTM types (e.g. Methylation) from being dominated by `Sites=<>` examples, which otherwise causes the model to collapse to always-empty predictions for those instructions. The validation fold is balanced as well so that `eval_loss` (and therefore the early-stopping signal and the best-checkpoint selection) tracks site-discrimination quality rather than the model's ability to memorize the empty prior — the natural-distribution val fold makes `eval_loss` a misleading proxy in this regime, since most of its targets are `Sites=<>`. Relative training abundance across PTM types still reflects the natural source distribution (Phosphorylation > Ubiquitination > Methylation); only the negative-to-positive ratio within each PTM type is capped.

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    bf16=True,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=LOGGING_STEPS,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    optim='adamw_torch',
    report_to='none',
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    gradient_checkpointing=False,
    seed=SPLIT_SEED,
)

response_template = '### Response:\n'
response_template_ids = tokenizer.encode('\n' + response_template, add_special_tokens=False)[2:]
print('response_template_ids:', response_template_ids)
print('decoded back        :', repr(tokenizer.decode(response_template_ids)))
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    args=sft_config,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

trainer.model.print_trainable_parameters()

## 9. Train

In [ ]:
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(os.path.join(OUTPUT_DIR, 'trainer_state.json'), 'w') as f:
    json.dump(trainer.state.__dict__, f, indent=2, default=str)
with open(os.path.join(OUTPUT_DIR, 'trainer_log_history.json'), 'w') as f:
    json.dump(trainer.state.log_history, f, indent=2)

print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best eval loss:', trainer.state.best_metric)

## 10. Plot Loss Curves

Training and evaluation loss curves are saved as a PNG for inclusion in the model card.

In [ ]:
log_history = trainer.state.log_history
train_logs = [(l['step'], l['loss']) for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_logs  = [(l['step'], l['eval_loss']) for l in log_history if 'eval_loss' in l]

fig, ax = plt.subplots(figsize=(9, 5))
if train_logs:
    s, v = zip(*train_logs)
    ax.plot(s, v, label='Train loss', alpha=0.7, linewidth=1.2)
if eval_logs:
    s, v = zip(*eval_logs)
    ax.plot(s, v, label='Eval loss', marker='o', linewidth=2)
if trainer.state.best_metric is not None and eval_logs:
    best_step = min(eval_logs, key=lambda x: x[1])[0]
    ax.axvline(best_step, color='gray', linestyle='--', alpha=0.5,
               label=f'Best checkpoint (step {best_step})')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Progress')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
loss_plot_path = os.path.join(OUTPUT_DIR, 'training_loss.png')
fig.savefig(loss_plot_path, dpi=150)
plt.show()
print('Saved:', loss_plot_path)

## 11. Sample Inference

Generate a completion from the trained adapter to confirm it produces output in the expected `Sites=<...>` format. A validation record is sampled and the model is prompted with that record's instruction; the predicted output is compared with the ground-truth sites for the window.

In [ ]:
model.eval()
sample = val_records[0]
prompt = ALPACA_TEMPLATE.format(instruction=sample['instruction'], input=sample['input'], output='')
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False, pad_token_id=tokenizer.pad_token_id)
completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print('PTM type :', sample['ptm_type'])
print('Input    :', sample['input'])
print('Expect   :', sample['output'])
print('Got      :', completion.strip())

## 12. Merge LoRA Adapter into the Base Model

We merge the LoRA weights back into the base model in fp16. This produces a single self-contained model that the evaluation notebook can load with one call to `AutoModelForCausalLM.from_pretrained`.

In [ ]:
del model, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

merged = AutoPeftModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    device_map='auto',
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='2GB')
tokenizer.save_pretrained(MERGED_DIR)
print('Merged model saved to', MERGED_DIR)
!ls -lh {MERGED_DIR}

## 13. Push the Merged Model to Hugging Face

This cell uploads the merged weights, tokenizer, loss plot, trainer logs, and the protein-split assignment. The evaluation notebook overwrites `README.md` after computing test metrics.

In [ ]:
import shutil

for fn in ['training_loss.png', 'trainer_log_history.json', 'trainer_state.json', 'protein_splits.json']:
    src = os.path.join(OUTPUT_DIR, fn)
    if os.path.exists(src):
        shutil.copy(src, MERGED_DIR)

stub_card = f'''---
base_model: {BASE_MODEL}
tags:
  - protein
  - ptm
  - methylation
  - phosphorylation
  - ubiquitination
  - lora
  - peft
library_name: transformers
---

# PTM-LLaMA

LoRA-fine-tuned `{BASE_MODEL}` for predicting post-translational modification (PTM) sites in protein sequences. A single LoRA adapter is instruction-tuned to handle three PTM types — methylation, phosphorylation, and ubiquitination — selected at inference time by the prompt. Output format: `Sites=<R5,D12,...>` regardless of PTM type.

This is a **training-only stub card**. Final metrics (per-PTM-type AUC, accuracy, precision, recall, F1, confusion matrices, and the cross-instruction ablation) are filled in by the companion evaluation notebook after running on the held-out test set.
'''
with open(os.path.join(MERGED_DIR, 'README.md'), 'w') as f:
    f.write(stub_card)

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=False)
api.upload_folder(
    folder_path=MERGED_DIR,
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
    delete_patterns='*',
    commit_message='Push PTM-LLaMA training artifacts',
)
print('Pushed to', f'https://huggingface.co/{HF_REPO_ID}')

## Next Steps

Proceed to the evaluation notebook (`evaluation/evaluate_ptm_llama.ipynb`) to compute per-PTM-type metrics, calibrate the operating thresholds, run the cross-instruction ablation, and finalize the model card.